# Demo: substitu?n? ?ifra a kryptoanal?za

C?lem projektu je uk?zat klasickou substitu?n? ?ifru nad abecedou `ABCDEFGHIJKLMNOPQRSTUVWXYZ_`, vytvo?en? referen?n? bigramov? matice z ?esk?ch text? a prolomen? ciphertextu pomoc? Metropolis-Hastings algoritmu.

Fin?ln? dlouh? b?h se spou?t? skriptem `scripts/decrypt_samples.py --iterations 20000`. Notebook slou?? jako p?ehledn? demonstrace a vyhodnocen?, proto v n?m nejsou extr?mn? dlouh? v?po?ty.

In [ ]:
from pathlib import Path
import csv
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from substitution_cipher import ALPHABET, substitute_encrypt, substitute_decrypt
from substitution_cipher.bigrams import get_bigrams, load_matrix
from substitution_cipher.cryptanalysis import plausibility, prolom_substitute

print(PROJECT_ROOT)
print(ALPHABET)

## Pou?it? abeceda

Projekt pou??v? p?esn? tuto abecedu:

```text
ABCDEFGHIJKLMNOPQRSTUVWXYZ_
```

Znak `_` reprezentuje mezeru. Referen?n? texty i ciphertexty se validuj? tak, aby obsahovaly pouze tyto znaky.

## ?ifrov?n? a de?ifrov?n?

Kl?? je permutace cel? abecedy. ?ifrov?n? mapuje p?vodn? abecedu na kl??, de?ifrov?n? pou??v? inverzn? mapov?n?.

In [ ]:
key = ALPHABET[3:] + ALPHABET[:3]
plaintext = 'BYL_POZDNI_VECER_PRVNI_MAJ'

ciphertext = substitute_encrypt(plaintext, key)
decrypted = substitute_decrypt(ciphertext, key)

print('Plaintext :', plaintext)
print('Ciphertext:', ciphertext)
print('Decrypted :', decrypted)
print('Round-trip OK:', decrypted == plaintext)

## Referen?n? texty

Pro jazykov? model ?e?tiny jsou pou?ity dva texty Karla ?apka:

- Krakatit: `data/processed/clean_text.txt`
- V?lka s mloky: `data/reference_texts/valka_s_mloky_clean.txt`

Skript `scripts/build_combined_reference_matrix.py` spoj? dostupn? texty p?es jeden znak `_`, ulo?? `data/processed/combined_clean_text.txt` a vytvo?? `data/processed/TM_ref.npy`.

In [ ]:
combined_text_path = PROJECT_ROOT / 'data' / 'processed' / 'combined_clean_text.txt'
matrix_path = PROJECT_ROOT / 'data' / 'processed' / 'TM_ref.npy'

combined_text = combined_text_path.read_text(encoding='utf-8').strip()
TM_ref = load_matrix(matrix_path)

print('D?lka spojen?ho textu:', len(combined_text))
print('Po?et bigram?:', len(get_bigrams(combined_text)))
print('Shape matice:', TM_ref.shape)
print('Suma matice:', TM_ref.sum())
print('Obsahuje nuly:', bool((TM_ref == 0).any()))

## Kontrola matice

Fin?ln? kontrola po sestaven? kombinovan? matice:

- shape: `(27, 27)`
- suma: `1.000000000000`
- obsahuje nuly: `False`
- d?lka spojen?ho referen?n?ho textu: `816372`
- po?et bigram?: `816371`

## Vizualizace bigramov? matice

N?sleduj?c? obr?zek ukazuje logaritmovan? hodnoty referen?n? bigramov? matice. Logaritmus je pou?it proto, aby byly rozd?ly v ?etnostech viditeln?j??.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(7, 6))
plt.imshow(np.log(TM_ref), cmap='viridis')
plt.colorbar(label='log pravd?podobnosti')
plt.xticks(range(len(ALPHABET)), list(ALPHABET), rotation=90)
plt.yticks(range(len(ALPHABET)), list(ALPHABET))
plt.title('Referen?n? bigramov? matice')
plt.tight_layout()
plt.show()

## Kr?tk? uk?zka kryptoanal?zy

Uk?zka pou??v? u?itelsk? soubor `text_1000_sample_1_ciphertext.txt`. Aby notebook z?stal rychl?, b??? zde pouze kr?tk? demonstra?n? po?et iterac?. Fin?ln? v?sledky v `outputs/` vznikly skriptem s `20 000` iteracemi.

In [ ]:
sample_ciphertext_path = PROJECT_ROOT / 'data' / 'ciphertexts' / 'text_1000_sample_1_ciphertext.txt'
sample_ciphertext = sample_ciphertext_path.read_text(encoding='utf-8').strip()

demo_key, demo_plaintext, demo_score = prolom_substitute(
    sample_ciphertext,
    TM_ref,
    iter=300,
    seed=1,
    progress_every=0,
)

print('D?lka ciphertextu:', len(sample_ciphertext))
print('Demo sk?re:', demo_score)
print('Uk?zka plaintextu:', demo_plaintext[:120])

## Porovn?n? s u?itelsk?m p??kladem

U?itelsk? p??klad pro `text_1000_sample_1` je ulo?en ve slo?ce `data/teacher_example/`. Fin?ln? v?stup ze skriptu je v `outputs/`.

In [ ]:
teacher_plaintext = (PROJECT_ROOT / 'data' / 'teacher_example' / 'text_1000_sample_1_plaintext.txt').read_text(encoding='utf-8').strip()
teacher_key = (PROJECT_ROOT / 'data' / 'teacher_example' / 'text_1000_sample_1_key.txt').read_text(encoding='utf-8').strip()
output_plaintext = (PROJECT_ROOT / 'outputs' / 'text_1000_sample_1_plaintext.txt').read_text(encoding='utf-8').strip()
output_key = (PROJECT_ROOT / 'outputs' / 'text_1000_sample_1_key.txt').read_text(encoding='utf-8').strip()

matching_chars = sum(1 for a, b in zip(output_plaintext, teacher_plaintext) if a == b)
matching_percent = matching_chars / len(teacher_plaintext) * 100

print('Plaintext p?esn? sed?:', output_plaintext == teacher_plaintext)
print('Spr?vn? znaky:', matching_chars)
print('Procento shody:', matching_percent)
print('Key p?esn? sed?:', output_key == teacher_key)
print('Nalezen? key je validn? d?lky:', len(output_key) == len(ALPHABET))

## Souhrn vyhodnocen?

Skript `scripts/evaluate_outputs.py` vytvo?il soubory:

- `reports/evaluation_summary.md`
- `reports/evaluation_summary.csv`

Tabulka obsahuje d?lku textu, sample ID, n?zvy v?stupn?ch soubor?, validitu kl??e, plausibility sk?re a porovn?n? s u?itelsk?m p??kladem tam, kde je zn?m? plaintext.

In [ ]:
evaluation_csv = PROJECT_ROOT / 'reports' / 'evaluation_summary.csv'
with evaluation_csv.open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))

teacher_row = next(row for row in rows if row['length'] == '1000' and row['sample_id'] == '1')

print('Po?et vyhodnocen?ch v?stup?:', len(rows))
print('Teacher sample row:')
for key, value in teacher_row.items():
    print(f'  {key}: {value}')

## Uk?zka exportovan?ch soubor?

V?stupy jsou ukl?d?ny samostatn? jako plaintext a key soubor. N?zvy dodr?uj? form?t po?adovan? v zad?n?.

In [ ]:
for path in sorted((PROJECT_ROOT / 'outputs').glob('text_1000_sample_1_*.txt')):
    content = path.read_text(encoding='utf-8').strip()
    print(path.name, '-> d?lka', len(content))
    print(content[:120])
    print()

## Z?v?r

Projekt ?sp??n? zpracoval v?ech `60` u?itelsk?ch ciphertext?. Fin?ln? b?h pou??v? `20 000` iterac? na ka?d? ciphertext a v?sledky ukl?d? do slo?ky `outputs/`.

Pro u?itelsk? sample `text_1000_sample_1` fin?ln? plaintext sed? p?esn?: `1000 / 1000` znak?, tedy `100 %`. Key soubor je validn? permutace abecedy; nen? textov? toto?n? s u?itelsk?m key souborem, proto?e n?kolik znak? se v tomto konkr?tn?m ciphertextu nevyskytuje a nejsou z n?j jednozna?n? ur?iteln?.